# Service Access — Silver
Water, electricity and household internet access · 2021–2025.

Run all cells in order. Set job maximum concurrent runs to **1**; use one writer across all jobs and keep Bronze unchanged during this run. IDs start at **4,000,001** and existing IDs remain unchanged. Missing values stay NULL; omitted observations are retained.

National records are separate from regional records. Percentages with different units must remain separate. Region codes align labels only: verify geographic boundaries before joining to budgets or GRDP (including NIR and BARMM/Sulu). Source notes remain available in Bronze; unverified missing reasons stay NULL.

## 1. Clean Bronze
Trim text, parse numbers and select the three measures.

In [ ]:
%sql
SET TIME ZONE 'UTC';

CREATE OR REPLACE TEMP VIEW service_access_clean AS
SELECT
    NULLIF(TRIM(country), '') AS country,
    TRIM(year) AS year_raw,
    CASE WHEN TRIM(year) RLIKE '^[0-9]{4}$'
         THEN TRY_CAST(TRIM(year) AS INT) END AS year,
    CASE TRIM(indicator)
        WHEN 'At least basic drinking-water access' THEN 'Water'
        WHEN 'Electricity access' THEN 'Electricity'
        WHEN 'Internet access at home' THEN 'Internet'
    END AS service,
    TRIM(indicator) AS service_measure,
    NULLIF(TRIM(region), '') AS source_region,
    NULLIF(TRIM(breakdown_type), '') AS breakdown_type,
    NULLIF(TRIM(breakdown_value), '') AS breakdown_value,
    NULLIF(TRIM(value), '') AS value_raw,
    TRY_CAST(NULLIF(TRIM(value), '') AS DECIMAL(18,10)) AS value,
    NULLIF(TRIM(unit), '') AS unit,
    CAST(NULL AS STRING) AS missing_reason,
    NULLIF(TRIM(source), '') AS source,
    NULLIF(TRIM(_run_id), '') AS run_id,
    _rescued_data
FROM pondolens.dev_bronze.service_access
WHERE TRIM(indicator) IN (
    'At least basic drinking-water access',
    'Electricity access',
    'Internet access at home'
);

## 2. Map regions
Use the same names and codes as GRDP. National uses PHL.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_region_map AS
SELECT * FROM VALUES
    ('BARMM', 'Bangsamoro Autonomous Region in Muslim Mindanao', 'BARMM'),
    ('CAR', 'Cordillera Administrative Region', 'CAR'),
    ('MIMAROPA', 'MIMAROPA Region', 'IVB'),
    ('NCR', 'National Capital Region', 'NCR'),
    ('National', 'Philippines', 'PHL'),
    ('Region I (Ilocos Region)', 'Ilocos Region', 'I'),
    ('Region II (Cagayan Valley)', 'Cagayan Valley', 'II'),
    ('Region III (Central Luzon)', 'Central Luzon', 'III'),
    ('Region IV-A (CALABARZON)', 'CALABARZON', 'IVA'),
    ('Region V (Bicol Region)', 'Bicol Region', 'V'),
    ('Region VI (Western Visayas)', 'Western Visayas', 'VI'),
    ('Region VII (Central Visayas)', 'Central Visayas', 'VII'),
    ('Region VIII (Eastern Visayas)', 'Eastern Visayas', 'VIII'),
    ('Region IX (Zamboanga Peninsula)', 'Zamboanga Peninsula', 'IX'),
    ('Region X (Northern Mindanao)', 'Northern Mindanao', 'X'),
    ('Region XI (Davao Region)', 'Davao Region', 'XI'),
    ('Region XII (SOCCSKSARGEN)', 'SOCCSKSARGEN', 'XII'),
    ('Region XIII (Caraga)', 'Caraga', 'XIII')
AS mapping(source_region, region, region_code);

CREATE OR REPLACE TEMP VIEW service_access_prepared AS
SELECT s.*, m.region, m.region_code
FROM service_access_clean s
LEFT JOIN service_access_region_map m
    ON s.source_region = m.source_region
WHERE s.year BETWEEN 2021 AND 2025;

## 3. Validate input
Stop on invalid records, empty batches or duplicate observations.

In [ ]:
%sql
-- Invalid years must not disappear through the scope filter.
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid source year.')
FROM service_access_clean WHERE year IS NULL;

SELECT ASSERT_TRUE(COUNT(*) > 0, 'No observations in scope.')
FROM service_access_prepared;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Invalid source records; review Bronze.')
FROM service_access_prepared
WHERE country IS NULL OR country <> 'Philippines'
   OR region IS NULL OR region_code IS NULL
   OR service IS NULL OR service_measure IS NULL
   OR source IS NULL OR run_id IS NULL OR unit IS NULL
   OR _rescued_data IS NOT NULL
   OR (value_raw IS NOT NULL AND value IS NULL)
   OR value < 0 OR value > 100
   OR NOT (
       (service = 'Water' AND unit IN (
           '% of families', '% of population',
           '% of rural population', '% of urban population'))
       OR (service = 'Electricity' AND unit IN (
           '% of population', '% of rural population', '% of urban population'))
       OR (service = 'Internet' AND unit = '% of households')
   );

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate source observation keys.')
FROM (
    SELECT country, year, service, service_measure, region_code, unit, breakdown_type, breakdown_value
    FROM service_access_prepared
    GROUP BY country, year, service, service_measure, region_code, unit, breakdown_type, breakdown_value
    HAVING COUNT(*) > 1
);

## 4. Create and validate target
Preserve existing rows and block invalid IDs or duplicate keys.

In [ ]:
%sql
SET TIME ZONE 'UTC';

CREATE TABLE IF NOT EXISTS pondolens.dev_silver.service_access (
    service_access_id BIGINT NOT NULL,
    country STRING,
    year INT,
    service STRING,
    service_measure STRING,
    region STRING,
    region_code STRING,
    unit STRING,
    breakdown_type STRING,
    breakdown_value STRING,
    value DECIMAL(18,10),
    missing_reason STRING,
    source STRING,
    run_id STRING,
    processed_at TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT service_access_id)
    AND COALESCE(COUNT_IF(service_access_id <= 4000000), 0) = 0,
    'Target IDs must be unique and greater than 4000000.'
)
FROM pondolens.dev_silver.service_access;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Duplicate target observation keys.')
FROM (
    SELECT country, year, service, service_measure, region_code, unit, breakdown_type, breakdown_value
    FROM pondolens.dev_silver.service_access
    GROUP BY country, year, service, service_measure, region_code, unit, breakdown_type, breakdown_value
    HAVING COUNT(*) > 1
);

SELECT ASSERT_TRUE(COUNT(*) = 0, 'Target has invalid required fields.')
FROM pondolens.dev_silver.service_access
WHERE country IS NULL OR year IS NULL OR service IS NULL
   OR service_measure IS NULL OR region IS NULL OR region_code IS NULL
   OR unit IS NULL OR source IS NULL OR run_id IS NULL
   OR processed_at IS NULL OR value < 0 OR value > 100;


## 5. Assign IDs
Reuse matched IDs. Allocate new IDs above the existing maximum.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_upsert AS
-- ROW_NUMBER applies separately to new and matched observations.
SELECT
    COALESCE(
        t.service_access_id,
        m.max_id + ROW_NUMBER() OVER (
            PARTITION BY t.service_access_id IS NULL
            ORDER BY
                s.country, s.year, s.service_measure,
                s.region_code, s.unit,
                s.breakdown_type, s.breakdown_value
        )
    ) AS service_access_id,
    s.country,
    s.year,
    s.service,
    s.service_measure,
    s.region,
    s.region_code,
    s.unit,
    s.breakdown_type,
    s.breakdown_value,
    s.value,
    s.missing_reason,
    s.source,
    s.run_id
FROM service_access_prepared s
LEFT JOIN pondolens.dev_silver.service_access t
    ON  s.country = t.country
    AND s.year = t.year
    AND s.service = t.service
    AND s.service_measure = t.service_measure
    AND s.region_code = t.region_code
    AND s.unit = t.unit
    AND s.breakdown_type <=> t.breakdown_type
    AND s.breakdown_value <=> t.breakdown_value
CROSS JOIN (
    SELECT GREATEST(
        COALESCE(MAX(service_access_id), 4000000),
        4000000
    ) AS max_id
    FROM pondolens.dev_silver.service_access
) m;

## 6. Merge
Insert new observations and update changed records. No deletes.

In [ ]:
%sql
MERGE INTO pondolens.dev_silver.service_access t
USING service_access_upsert s
ON t.service_access_id = s.service_access_id

WHEN MATCHED AND (
       NOT (t.value <=> s.value)
    OR NOT (t.region <=> s.region)
    OR NOT (t.missing_reason <=> s.missing_reason)
    OR NOT (t.source <=> s.source)
    OR NOT (t.run_id <=> s.run_id)
)
THEN UPDATE SET
    t.value = s.value,
    t.region = s.region,
    t.missing_reason = s.missing_reason,
    t.source = s.source,
    t.run_id = s.run_id,
    t.processed_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    service_access_id,
    country, year, service, service_measure,
    region, region_code, unit,
    breakdown_type, breakdown_value,
    value, missing_reason, source, run_id, processed_at
)
VALUES (
    s.service_access_id,
    s.country, s.year, s.service, s.service_measure,
    s.region, s.region_code, s.unit,
    s.breakdown_type, s.breakdown_value,
    s.value, s.missing_reason, s.source, s.run_id,
    CURRENT_TIMESTAMP()
);

## 7. Verify
Check unique IDs and reconciliation with the prepared batch.

In [ ]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT service_access_id),
    'Duplicate IDs after MERGE.')
FROM pondolens.dev_silver.service_access;

-- Match business keys directly; do not recalculate IDs for reconciliation.
SELECT ASSERT_TRUE(COUNT(*) = 0, 'Batch reconciliation failed.')
FROM service_access_prepared s
LEFT JOIN pondolens.dev_silver.service_access t
    ON s.country = t.country AND s.year = t.year
    AND s.service = t.service AND s.service_measure = t.service_measure
    AND s.region_code = t.region_code AND s.unit = t.unit
    AND s.breakdown_type <=> t.breakdown_type
    AND s.breakdown_value <=> t.breakdown_value
WHERE t.service_access_id IS NULL
   OR NOT (s.value <=> t.value)
   OR NOT (s.region <=> t.region)
   OR NOT (s.source <=> t.source)
   OR NOT (s.run_id <=> t.run_id)
   OR NOT (s.missing_reason <=> t.missing_reason);

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT service_access_id) AS unique_ids,
    MIN(service_access_id) AS first_id,
    MAX(service_access_id) AS last_id,
    COUNT(value) AS available_values,
    COUNT_IF(value IS NULL) AS missing_values
FROM pondolens.dev_silver.service_access;

## 8. Publish access views
Gold reads available values. Missing observations remain in Silver for review.


In [ ]:
%sql
CREATE OR REPLACE VIEW pondolens.dev_silver.service_access_available AS
SELECT
    service_access_id, country, year, service, service_measure,
    region, region_code, unit, breakdown_type, breakdown_value,
    value, missing_reason, source, run_id, processed_at
FROM pondolens.dev_silver.service_access
WHERE value IS NOT NULL;

CREATE OR REPLACE VIEW pondolens.dev_silver.service_access_missing AS
SELECT
    service_access_id, country, year, service, service_measure,
    region, region_code, unit, breakdown_type, breakdown_value,
    value, missing_reason, source, run_id, processed_at,
    'MISSING_VALUE' AS quality_flag
FROM pondolens.dev_silver.service_access
WHERE value IS NULL;


## 9. Check coverage
Available and missing counts must equal the Silver total. Rerun unchanged inputs to confirm zero MERGE inserts and updates.


In [ ]:
%sql
SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM pondolens.dev_silver.service_access) =
    (SELECT COUNT(*) FROM pondolens.dev_silver.service_access_available) +
    (SELECT COUNT(*) FROM pondolens.dev_silver.service_access_missing),
    'Access view reconciliation failed.'
);

SELECT 'AVAILABLE' AS status, COUNT(*) AS total_rows
FROM pondolens.dev_silver.service_access_available
UNION ALL
SELECT 'MISSING_VALUE', COUNT(*)
FROM pondolens.dev_silver.service_access_missing;
